# Diffusion models: generating digits on command

In this notebook we build a small **denoising diffusion model** (DDPM) that generates MNIST digits, and we tell it *which* digit to draw.

The idea in one paragraph: we take real images and destroy them step by step with Gaussian noise. A U-Net is trained to look at a noisy image and say *what noise was added*. To generate a new image we start from pure noise and use the network to remove a little noise at a time, about a thousand times. Giving the network the digit label as an extra input lets us choose what comes out.

Contents
1. Data
2. The forward process: adding noise
3. The denoiser: a U-Net with time and label inputs
4. Training: this is just regression
5. Why not denoise in one step?
6. Sampling: from noise to a digit
7. Classifier-free guidance
8. Fewer steps (DDIM)
9. Try it yourself

Run on a GPU (Colab: *Runtime → Change runtime type → T4 GPU*). On a CPU the training is too slow.

In [ ]:
# Cell 1: Setup and imports
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import keras
from keras import layers, ops
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.config.backend()}")

## 1. Data

We scale pixel values to $[-1, 1]$, so that clean images and the Gaussian noise live on a similar scale. We also pad the $28\times28$ images to $32\times32$, so that the U-Net can halve the resolution three times ($32 \to 16 \to 8 \to 4$).

In [ ]:
# Cell 2: Load MNIST
(x_train, y_train), _ = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 127.5 - 1.0                            # [0, 255] -> [-1, 1]
x_train = np.pad(x_train, ((0, 0), (2, 2), (2, 2)), constant_values=-1.0)  # 28x28 -> 32x32
x_train = x_train[..., None]                                                # add channel axis
y_train = y_train.astype("int32")

NUM_CLASSES = 10
NULL_LABEL = NUM_CLASSES   # extra label 10 means "no label" (needed for guidance, section 7)

print(x_train.shape, y_train.shape, x_train.min(), x_train.max())

In [ ]:
# Cell 2b: Helper for showing images
def to_img(x):
    # [-1, 1] -> [0, 1], drop channel axis
    return np.clip((x[..., 0] + 1) / 2, 0, 1)

def show_grid(images, nrows, ncols, titles=None, figsize=None, suptitle=None):
    figsize = figsize or (ncols * 1.1, nrows * 1.1)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < len(images):
            ax.imshow(to_img(images[i]), cmap="gray", vmin=0, vmax=1)
            if titles is not None:
                ax.set_title(titles[i], fontsize=8)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

show_grid(x_train[:20], 2, 10, titles=[str(l) for l in y_train[:20]])

## 2. The forward process: adding noise

We use $T = 1000$ noise levels. At step $t$ we add a small amount of noise with variance $\beta_t$, where $\beta_t$ grows linearly from $10^{-4}$ to $0.02$. Write $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \alpha_1\alpha_2\cdots\alpha_t$.

The useful fact is that we never need to add noise step by step. A noisy image at any level $t$ can be produced directly from the clean image $x_0$:

$$
x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \varepsilon, \qquad \varepsilon \sim \mathcal N(0, I).
$$

So $x_t$ is a mix of the image and noise. As $t$ grows, the image weight $\sqrt{\bar\alpha_t}$ goes from 1 to almost 0 and the noise weight goes from 0 to almost 1. At $t = T$ the image is pure noise.

In [ ]:
# Cell 3: Noise schedule
T = 1000
betas = np.linspace(1e-4, 0.02, T).astype("float32")
alphas = 1.0 - betas
alpha_bars = np.cumprod(alphas).astype("float32")

def add_noise(x0, t, eps):
    # x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * eps, for a batch of timesteps t
    a = alpha_bars[t].reshape(-1, 1, 1, 1)
    return np.sqrt(a) * x0 + np.sqrt(1.0 - a) * eps

plt.figure(figsize=(6, 3))
plt.plot(np.sqrt(alpha_bars), label=r"image weight $\sqrt{\bar\alpha_t}$")
plt.plot(np.sqrt(1 - alpha_bars), label=r"noise weight $\sqrt{1-\bar\alpha_t}$")
plt.xlabel("t"); plt.legend(); plt.grid(); plt.show()

In [ ]:
# Cell 4: One digit at increasing noise levels
rng = np.random.default_rng(0)
steps_to_show = [0, 50, 100, 200, 300, 400, 500, 700, 999]
x0 = np.repeat(x_train[:1], len(steps_to_show), axis=0)
eps = rng.standard_normal(x0.shape).astype("float32")
xt = add_noise(x0, np.array(steps_to_show), eps)
show_grid(xt, 1, len(steps_to_show), titles=[f"t={t}" for t in steps_to_show])

## 3. The denoiser: a U-Net with time and label inputs

The network gets three inputs:

* the noisy image $x_t$,
* the noise level $t$ (how noisy the image is),
* the digit label $y$ (what the image should become). Label 10 means "no label".

It outputs its estimate $\hat\varepsilon$ of the noise that was added. The output has the same shape as the image.

The architecture is the same kind of U-Net as in the image-repair example: an encoder that halves the resolution, a decoder that doubles it back, and skip connections between them. Two additions:

* **Time embedding.** The number $t$ is turned into a vector with sine and cosine functions of different frequencies (the same trick as positional encoding in Transformers).
* **Label embedding.** The label goes through an `Embedding` layer.

The two vectors are concatenated and passed through two dense layers, and the result is added to the feature maps inside every residual block. This way every layer knows how noisy the input is and which digit we want.

In [ ]:
# Cell 5: Model
@keras.saving.register_keras_serializable()
class SinusoidalEmbedding(layers.Layer):
    # Maps a timestep t (shape (batch, 1)) to a vector of sines and cosines (shape (batch, dim))
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim

    def call(self, t):
        half = self.dim // 2
        freqs = ops.exp(-np.log(10000.0) * ops.arange(half, dtype="float32") / half)
        args = ops.cast(t, "float32") * freqs[None, :]
        return ops.concatenate([ops.sin(args), ops.cos(args)], axis=-1)

    def get_config(self):
        return {**super().get_config(), "dim": self.dim}


def res_block(x, emb, channels):
    skip = x if x.shape[-1] == channels else layers.Conv2D(channels, 1)(x)
    h = layers.GroupNormalization(groups=8)(x)
    h = layers.Activation("swish")(h)
    h = layers.Conv2D(channels, 3, padding="same")(h)
    # Inject time + label information: one number per channel, broadcast over the image
    e = layers.Dense(channels)(emb)
    h = h + layers.Reshape((1, 1, channels))(e)
    h = layers.GroupNormalization(groups=8)(h)
    h = layers.Activation("swish")(h)
    h = layers.Conv2D(channels, 3, padding="same")(h)
    return layers.Add()([h, skip])


def build_unet(widths=(32, 64, 128), emb_dim=128):
    image_in = keras.Input(shape=(32, 32, 1), name="noisy_image")
    t_in = keras.Input(shape=(1,), name="timestep")
    label_in = keras.Input(shape=(1,), dtype="int32", name="label")

    # Time and label embeddings, combined into one conditioning vector
    t_emb = SinusoidalEmbedding(emb_dim)(t_in)
    t_emb = layers.Dense(emb_dim, activation="swish")(t_emb)
    # Label embedding; initialised on the same scale as the time features, so that
    # the network notices the label from the start of training
    y_emb = layers.Embedding(NUM_CLASSES + 1, emb_dim,
                             embeddings_initializer=keras.initializers.RandomNormal(stddev=1.0))(label_in)
    y_emb = layers.Flatten()(y_emb)
    emb = layers.Concatenate()([t_emb, y_emb])
    emb = layers.Dense(emb_dim, activation="swish")(emb)
    emb = layers.Dense(emb_dim, activation="swish")(emb)

    x = layers.Conv2D(widths[0], 3, padding="same")(image_in)

    # Encoder: 32 -> 16 -> 8 -> 4
    skips = []
    for w in widths:
        x = res_block(x, emb, w)
        skips.append(x)
        x = layers.Conv2D(w, 3, strides=2, padding="same")(x)

    # Bottleneck at 4x4
    x = res_block(x, emb, widths[-1])
    x = res_block(x, emb, widths[-1])

    # Decoder: 4 -> 8 -> 16 -> 32, with skip connections
    for w, s in zip(reversed(widths), reversed(skips)):
        x = layers.UpSampling2D(2)(x)
        x = layers.Concatenate()([x, s])
        x = res_block(x, emb, w)
        x = res_block(x, emb, w)

    x = layers.GroupNormalization(groups=8)(x)
    x = layers.Activation("swish")(x)
    noise_out = layers.Conv2D(1, 3, padding="same", kernel_initializer="zeros")(x)

    return keras.Model([image_in, t_in, label_in], noise_out, name="denoiser")

model = build_unet()
print(f"Parameters: {model.count_params():,}")

## 4. Training: this is just regression

A training example is made like this:

1. take a real image $x_0$ and its label $y$,
2. pick a random noise level $t \in \{0, \dots, T-1\}$,
3. draw noise $\varepsilon$ and compute $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \varepsilon$,
4. the input is $(x_t, t, y)$ and the target is $\varepsilon$.

The loss is the mean squared error $\lVert \varepsilon - \hat\varepsilon(x_t, t, y) \rVert^2$. That is the whole training algorithm. The new batches are made on the fly, so every epoch sees new noise.

In 10% of the examples we replace the label with "no label" (10). Then the same network also learns to denoise *without* knowing the digit. We need that in section 7.

The optimizer keeps an exponential moving average (EMA) of the weights (`use_ema=True`), and the averaged weights are used after training. This is standard for diffusion models and gives noticeably cleaner samples.

In [ ]:
# Cell 6: Training data generator
class NoisyBatches(keras.utils.PyDataset):
    def __init__(self, x, y, batch_size=128, p_uncond=0.1, seed=0, **kwargs):
        super().__init__(**kwargs)
        self.x, self.y = x, y
        self.batch_size = batch_size
        self.p_uncond = p_uncond
        self.rng = np.random.default_rng(seed)
        self.perm = self.rng.permutation(len(x))

    def __len__(self):
        return len(self.x) // self.batch_size

    def __getitem__(self, i):
        idx = self.perm[i * self.batch_size:(i + 1) * self.batch_size]
        x0 = self.x[idx]
        y = self.y[idx].copy()
        y[self.rng.random(len(y)) < self.p_uncond] = NULL_LABEL   # sometimes hide the label
        t = self.rng.integers(0, T, size=len(y))
        eps = self.rng.standard_normal(x0.shape).astype("float32")
        xt = add_noise(x0, t, eps)
        inputs = {"noisy_image": xt,
                  "timestep": t[:, None].astype("float32"),
                  "label": y[:, None]}
        return inputs, eps

    def on_epoch_end(self):
        self.perm = self.rng.permutation(len(self.x))

train_data = NoisyBatches(x_train, y_train, batch_size=128)

In [ ]:
# Cell 7: Train
EPOCHS = 20   # check how long the first epoch takes, and reduce if needed

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3, use_ema=True, ema_momentum=0.99),
    loss="mse",
)
history = model.fit(train_data, epochs=EPOCHS)

plt.plot(history.history["loss"])
plt.xlabel("epoch"); plt.ylabel("MSE of predicted noise"); plt.grid(); plt.show()

In [ ]:
# Cell 8: Save / load weights
model.save_weights("diffusion_mnist.weights.h5")
# In a new session: run cells 1-5, then
# model.load_weights("diffusion_mnist.weights.h5")

## 5. Why not denoise in one step?

From the predicted noise we can always compute the network's guess of the clean image, by solving the forward formula for $x_0$:

$$
\hat x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\hat\varepsilon}{\sqrt{\bar\alpha_t}}.
$$

So why not start from pure noise and jump straight to $\hat x_0$?

The network is trained with mean squared error. The best MSE answer is the **average** of all images that could have produced this noisy input. From pure noise, every "3" is equally plausible, so the one-step answer is roughly the average "3": a blurry blob. An image-repair U-Net trained with MSE behaves the same way when its input is pure noise.

Diffusion avoids this by taking a small step towards that average, adding back a little fresh noise, and asking again. Each step commits to a few more details, and the guesses become sharper and sharper.

In [ ]:
# Cell 9: Helper that predicts the noise (used by all samplers below)
def predict_eps(x, t, labels, guidance=1.0):
    # Noise prediction at timestep t for a batch x with the given labels.
    # guidance = 1: plain conditional model.  guidance = 0: unconditional model.
    # guidance > 1: classifier-free guidance (section 7).
    n = len(x)
    t_arr = np.full((n, 1), t, dtype="float32")
    cond = labels.reshape(-1, 1).astype("int32")
    uncond = np.full((n, 1), NULL_LABEL, dtype="int32")
    if guidance == 1.0:
        return model.predict_on_batch([x, t_arr, cond])
    if guidance == 0.0:
        return model.predict_on_batch([x, t_arr, uncond])
    # One forward pass for both: stack conditional and unconditional copies
    eps = model.predict_on_batch([np.concatenate([x, x]),
                                  np.concatenate([t_arr, t_arr]),
                                  np.concatenate([cond, uncond])])
    eps_c, eps_u = eps[:n], eps[n:]
    return eps_u + guidance * (eps_c - eps_u)

def predict_x0(x, t, eps):
    a = alpha_bars[t]
    return (x - np.sqrt(1 - a) * eps) / np.sqrt(a)

In [ ]:
# Cell 10: One-step denoising from pure noise vs. real digits
rng = np.random.default_rng(1)
labels = np.arange(10)
x_t = rng.standard_normal((10, 32, 32, 1)).astype("float32")   # pure noise

# We tell the model this is noise level t = 600, where the image weight is only 0.16,
# so pure noise is a realistic input. (At t = 999 the weight is 0.006, and dividing
# by it in predict_x0 would blow up even tiny errors of the network.)
t = 600
eps = predict_eps(x_t, t, labels)
x0_one_step = np.clip(predict_x0(x_t, t, eps), -1, 1)

show_grid(x0_one_step, 1, 10, titles=[str(l) for l in labels],
          suptitle="One-step guess of the clean image from pure noise")

## 6. Sampling: from noise to a digit

Sampling runs the process backwards. We start from pure noise $x_T \sim \mathcal N(0, I)$ and for $t = T-1, \dots, 0$:

1. predict the noise $\hat\varepsilon$ and from it the clean-image guess $\hat x_0$,
2. move to a slightly less noisy image $x_{t-1}$, which is a weighted mix of $\hat x_0$ and the current $x_t$, plus a little fresh noise:

$$
x_{t-1} = \frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,\hat x_0 \;+\; \frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,x_t \;+\; \sigma_t z, \qquad \sigma_t^2 = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t.
$$

This is Algorithm 2 in Ho et al. (2020), written with $\hat x_0$ instead of $\hat\varepsilon$ (the two forms are algebraically equal). Writing it this way lets us clip $\hat x_0$ to the valid pixel range $[-1, 1]$, which helps a lot with a small model.

The random starting noise works as the **seed**: a different seed gives a different "7".

In [ ]:
# Cell 11: DDPM sampler
def sample_ddpm(labels, guidance=1.0, seed=None, record_every=None):
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((len(labels), 32, 32, 1)).astype("float32")
    frames = []   # (t, x_t, x0_hat) snapshots for visualisation
    for t in reversed(range(T)):
        eps = predict_eps(x, t, labels, guidance)
        x0 = np.clip(predict_x0(x, t, eps), -1, 1)
        a_bar, a_bar_prev = alpha_bars[t], (alpha_bars[t - 1] if t > 0 else 1.0)
        coef_x0 = np.sqrt(a_bar_prev) * betas[t] / (1 - a_bar)
        coef_xt = np.sqrt(alphas[t]) * (1 - a_bar_prev) / (1 - a_bar)
        x = coef_x0 * x0 + coef_xt * x
        if t > 0:
            sigma = np.sqrt((1 - a_bar_prev) / (1 - a_bar) * betas[t])
            x = x + sigma * rng.standard_normal(x.shape).astype("float32")
        if record_every and (t % record_every == 0 or t == T - 1):
            frames.append((t, x.copy(), x0.copy()))
    return (x, frames) if record_every else x

In [ ]:
# Cell 12: Ten of each digit
labels = np.repeat(np.arange(10), 10)
samples = sample_ddpm(labels, guidance=1.0, seed=42)
show_grid(samples, 10, 10, figsize=(8, 8), suptitle="Rows: digits 0-9, columns: different random seeds")

In [ ]:
# Cell 13: Watch the denoising: x_t (top) and the model's current guess x0_hat (bottom)
labels = np.array([3])
_, frames = sample_ddpm(labels, seed=7, record_every=1)
show_t = [999, 900, 800, 700, 600, 500, 400, 300, 200, 100, 0]
by_t = {t: (xt, x0) for t, xt, x0 in frames}
row_xt = [by_t[t][0][0] for t in show_t]
row_x0 = [by_t[t][1][0] for t in show_t]
show_grid(row_xt + row_x0, 2, len(show_t), titles=[f"t={t}" for t in show_t] + [""] * len(show_t),
          suptitle=r"Top: noisy image $x_t$.   Bottom: guess of the final image $\hat x_0$")

In [ ]:
# Cell 14: Animation of 16 samples being generated
labels = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5])
_, frames = sample_ddpm(labels, seed=3, record_every=20)

def tile(batch, ncols=8):
    imgs = to_img(batch)
    rows = [np.concatenate(list(imgs[i:i + ncols]), axis=1) for i in range(0, len(imgs), ncols)]
    return np.concatenate(rows, axis=0)

fig, ax = plt.subplots(figsize=(8, 2.2))
ax.axis("off")
im = ax.imshow(tile(frames[0][1]), cmap="gray", vmin=0, vmax=1)
title = ax.set_title("")

def update(i):
    t, xt, _ = frames[i]
    im.set_data(tile(xt))
    title.set_text(f"t = {t}")
    return im, title

anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=80)
plt.close(fig)
HTML(anim.to_jshtml())

## 7. Classifier-free guidance

Because 10% of the training labels were hidden, the network can give two noise predictions for the same image: a conditional one $\hat\varepsilon(x_t, t, y)$ and an unconditional one $\hat\varepsilon(x_t, t, \varnothing)$. Their difference points in the direction that makes the image "more like the label". Guidance moves further in that direction:

$$
\hat\varepsilon_{\text{guided}} = \hat\varepsilon(x_t, t, \varnothing) + s\,\bigl(\hat\varepsilon(x_t, t, y) - \hat\varepsilon(x_t, t, \varnothing)\bigr).
$$

* $s = 0$: the label is ignored and we get random digits (an unconditional model).
* $s = 1$: the plain conditional model.
* $s > 1$: the digits follow the label more strongly and look more "typical", but they also become more alike.

The `guidance_scale` setting in Stable Diffusion is exactly this $s$, with a text prompt in place of the label.

In [ ]:
# Cell 15: Same seeds, different guidance scales
labels = np.tile(np.arange(10), 3)   # digits 0-9, three times
for s in [0.0, 1.0, 3.0, 7.0]:
    samples = sample_ddpm(labels, guidance=s, seed=11)
    show_grid(samples, 3, 10, figsize=(8, 2.6), suptitle=f"guidance scale s = {s}   (columns are labels 0-9)")

## 8. Fewer steps (DDIM)

A thousand network calls per image is slow. **DDIM** (Song et al., 2021) uses the *same trained network* but takes big jumps: from $x_t$ it computes $\hat x_0$ and $\hat\varepsilon$ and goes directly to an earlier time $t'$,

$$
x_{t'} = \sqrt{\bar\alpha_{t'}}\,\hat x_0 + \sqrt{1-\bar\alpha_{t'}}\,\hat\varepsilon,
$$

with no fresh noise. This is the forward formula again, with the network's guesses in place of the true image and noise. The process is deterministic, so the starting noise alone fixes the image. The "number of steps" setting in image generators is this.

DDIM is more demanding on the network than DDPM. In DDPM the fresh noise added at every step washes out small mistakes; DDIM has no such correction, so errors of the network stay in the image. If the DDIM samples look noisy or broken while the DDPM samples in section 6 look fine, the model needs more training.

In [ ]:
# Cell 16: DDIM sampler
def sample_ddim(labels, steps=50, guidance=1.0, seed=None, x_start=None):
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)
    x = x_start.copy() if x_start is not None else \
        rng.standard_normal((len(labels), 32, 32, 1)).astype("float32")
    ts = np.linspace(T - 1, 0, steps).round().astype(int)
    for i, t in enumerate(ts):
        eps = predict_eps(x, t, labels, guidance)
        x0 = np.clip(predict_x0(x, t, eps), -1, 1)
        eps = (x - np.sqrt(alpha_bars[t]) * x0) / np.sqrt(1 - alpha_bars[t])   # consistent with clipped x0
        a_prev = alpha_bars[ts[i + 1]] if i + 1 < len(ts) else 1.0
        x = np.sqrt(a_prev) * x0 + np.sqrt(1 - a_prev) * eps
    return x

In [ ]:
# Cell 17: Same starting noise, different numbers of steps
import time
labels = np.arange(10)
x_start = np.random.default_rng(5).standard_normal((10, 32, 32, 1)).astype("float32")
for steps in [3, 10, 25, 100]:
    t0 = time.time()
    samples = sample_ddim(labels, steps=steps, x_start=x_start)
    show_grid(samples, 1, 10, figsize=(8, 1.2), suptitle=f"DDIM, {steps} steps ({time.time() - t0:.1f} s)")

## 9. Try it yourself

1. **Fashion-MNIST.** Replace `mnist` with `fashion_mnist` in cell 2 and retrain. Labels are now clothing types.
2. **Train less.** Train for 1, 3 and 10 epochs. At what point do the digits become recognisable?
3. **No guidance training.** Set `p_uncond=0.0`. What happens to guidance in cell 15, and why?
4. **Interpolate.** In cell 16, use the same starting noise but a *mix* of two label embeddings. (Hint: build a model that takes the embedding vector instead of the label.)
5. **CIFAR-10.** Change the input to $32\times32\times3$ and train on CIFAR-10. Expect rough images: this needs a bigger model and much more training, which is why real image generators are expensive.

### From here to Stable Diffusion

Stable Diffusion is this notebook with three changes:

* the label is replaced by a **text embedding** of the prompt, and the U-Net reads it with **cross-attention** instead of adding one vector,
* the diffusion runs in the **latent space of an autoencoder** (64×64×4 instead of 512×512×3 pixels), and the decoder turns the final latent into an image,
* everything is much bigger and trained on hundreds of millions of image–caption pairs.

References: Ho, Jain & Abbeel, *Denoising Diffusion Probabilistic Models* (2020); Song, Meng & Ermon, *Denoising Diffusion Implicit Models* (2021); Ho & Salimans, *Classifier-Free Diffusion Guidance* (2022); Rombach et al., *High-Resolution Image Synthesis with Latent Diffusion Models* (2022).